# Day 26: 按长度分桶的真实执行引擎

Cache position 和本次输入长度相同的请求可以 stack 后调用一次模型。BatchCache 拆分每层更新回不同请求；prompt 不同但长度相同可以合批。

参考源码：`learning/mini_vllm/engine.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY22_30.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 26完整讲义](../docs/DAY26_BATCHED_ENGINE.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

同批加工可以共用模型调用，但不能把不同人的历史串成一句话：

```text
A历史 [1,Hkv,3,D] ─┐
B历史 [1,Hkv,3,D] ─┴→ Batch [2,Hkv,3,D]
                          ↓ 一次model forward
                    按Batch行拆回A/B
```

A/B的prompt内容不同但同长，可一起prefill。
若A已缓存3位置、B刚到需要prefill 2位置，
本实现按 `(cache_position,input_length)` 分桶，同一tick调用两次不同桶，
不是强行stack形状不兼容的请求。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(26, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

为什么不同位置的请求不能不带 mask/位置处理就直接 stack？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：两条用户历史能沿T拼接吗？**
答：不能这样冒充batch，它会改变上下文语义；应沿B并保证请求隔离。

**问：一个tick有多个model调用正常吗？**
答：在此分桶实现正常，一个桶一次调用，不同桶分开。

**问：P=3、G=3为什么预留5位置？**
答：prefill给首输出，后续只反馈前G-1个输出，最终token不再次forward。

**问：合批一定更快吗？**
答：不一定，小模型与gather/调度成本可能抵消收益，需要Day 27测量。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY26_BATCHED_ENGINE.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。